# Forming the Index of websites for our ML periodical analysis
#### Welcome to the first step!
Hi there! You made it to the first step of the data preprocessing processs for ML classification of WARC records for the XS4ALL web collection of the KB National Library in the Netherlands. In our specific case, we aim to predict the period the website might originate from, drawing from their **structural materialities**: a.k.a. the archived source code! We find this in the **WARC**. Wikipedia describes the WARC as:
> "The WARC (Web ARChive) archive format specifies a method for combining multiple digital resources into an aggregate archive file together with related information. These combined resources are saved as a WARC file which can be replayed using appropriate software such as ReplayWeb.page, or used by archive websites such as the Wayback Machine." (Accessed latest on June 1, 2026)

If you're new to this file format, there are many online resources available to increase your understanding such as the explanation of Archive-It: https://archive.org/details/introduction-to-the-warc. 

Once you familiarized yourself with the standards, we will move on to creating the index of URLs used for our ML classification pipeline.

#### This is **not** a Tool
Yet before moving forward, I wish to disclaim that the purposes of this notebook is **not** to provide a general tool for WARC exploration. There are many wonderful packages and libraries that can help you along your WARC journey (links stolen from helpful Rebecca Cremona of the Harvard Library Innovation Lab):
- [WARCbench](https://github.com/harvard-lil/warcbench)
- [Warcheology](https://github.com/nlnwa/warchaeology)
- [WARCAT](https://github.com/chfoo/warcat)
- [WARCIO](https://github.com/webrecorder/warcio)
- [warc](https://github.com/internetarchive/warc)

We will, however, make use of these tools for our specific research needs. 

This notebook (and all following code) is created for reproducability and transparency reasons. Jupyter notebooks is a text editor that lends itself to exchanging code and explaining decisions along the way. The periodical classification of the KB's XS4ALL webcollection is a highly specific case. Therefore, it requires a prudential record of all steps and information that is necessary to comprehend the choices that had needed to be made.

#### Index for creating the index
Although the notebook is part of a larger consortium of code available in the [GitHub repository](...), we will carefully begin with exploration of the various data and metadata we have at hand. The notebook is structured in the following order:
1. What is in the archive and what does the metadata describe?
2. Creating new metadata
3. Preparing the json training files

Let's start!

## 1. What is inside the archive and what does the metadata describe?
<div class="alert alert-block alert-info">
<b>Tip:</b> Read the collection description of the XS4ALL collection for understanding the collection curation and the specific circumstances that had decisive consequences, written by former Digital Conservator Kees Tsezelsky and available on Zenodo: .... [INPUT]
</div>

### Metadata
During the acquisition of the collection through the Web Curator Tool, the 'year' was manually annotated by the collection staff by inspecting the page-info and referring to 'last updated: day xx month xxxxx xx:xx:xx'. This was documented in the CSV '2020.10.31_xs4all_actief_selectie_masterbestand_gezamelijk_SH' along with more information that we will extract: 'ID', 'URL', 'Bijzonderheden/trefwoorden'.

Later, we will try to add the metadata 'Subject' and 'Type' added in the WCT tool for the purpose of the Knowledge Graph.

### Archive index
The other CSV 'kb-webcollectie_xs4all.csv' is published online via 'https://www.kb.nl/sites/default/files/documents/kb-webcollectie_xs4all.xlsx' and regularly (if needed) updated. The collection XS4ALL is quite fixed, so changes are not expected. This gives us information of all the webpages which are truely archived in the collection.


The metadata file is rich, yet mentions URLs non-existent in the /archive. Therefore, we must intersect the data from both CSV files with their URL.

![vendiagram-intersection-index-metadata](./img/data_pre-img1.png)
Figure: Screenshot of Miro board 'Thesis Internship XS4ALL' brainstorming on data processing steps.

*Goal of this notebook:*
- DF ['ID', 'URL', 'Year']
- DF ['ID', 'URL', 'Year', 'Bijzonderheden/trefwoorden', 'Subject', 'Type']
- my_dict {'URL': 'xxxx'} (hash map)

### 1. Loading the files
We load two CSVs containing:
1) URLs described in the curator metadata excel;
2) URLs actually present in the archive 

#### 1.1 Metadata csv

In [18]:
import pandas as pd

# Import 'masterbestand' and define Int64 datatypes to ignore NaN cells in ID and jaar/year columns
df_meta = pd.read_csv('../metadata/2020.10.31_xs4all_actief_selectie_masterbestand_gezamelijk_SH.csv', 
                encoding='latin-1', # the file had an older encoding
                sep=',',
                dtype={"ID": "Int64", 
                       "zonder https": "string",
                       "jaar": "Int64",
                       "URL": "string",
                       "Bijzonderheden/trefwoorden (altijd XS4ALL)": "string"}
                       )
# Translate a few colums from Dutch to English
df_meta.rename(columns={"zonder https": "no_https",
                         "jaar": "year",
                          'URL': 'url',
                         "Bijzonderheden/trefwoorden (altijd XS4ALL)": "description"}, 
                 inplace=True)

# Cut off unnecessary metadata for now.
# ID is not related to a WARC ID, so we will remove it too.
df_meta = df_meta[['ID', 'year', 'url', 'description']].copy()

In [19]:
# check imported dataframe
df_meta.head()

,ID,year,url,description
0,4940,2008,https://fambok.home.xs4all.nl/,site voor twee honden
1,971,2008,https://asz00457.home.xs4all.nl/,Wil & Betty's sabbatical
2,7634,2008,https://hsol.home.xs4all.nl/,CV
3,18844,2008,https://vmarja.home.xs4all.nl/,2002
4,14955,2008,https://rves.home.xs4all.nl/,"2004, wendy & robert homepage, persoonlijke pa..."


#### 1.2 Dropping duplicates
To check the number of duplicates in the master_bestand. We assume here that the URL can function as a unique identifier for each webpage.

In [20]:
print(f"Total number of records in the master_meta dataframe: {len(df_meta)}")
print(f"Unique URLs in the master_meta dataframe: {df_meta['url'].nunique()}", "\n")

meta_duplicates = df_meta[df_meta.duplicated('url', keep=False)]
print(f"Number of duplicates: {len(meta_duplicates)}")
print(meta_duplicates.sort_values('url')[:10])

Total number of records in the master_meta dataframe: 18848
Unique URLs in the master_meta dataframe: 18837 

Number of duplicates: 22
          ID  year                               url  \
15404    769  2010       https://are.home.xs4all.nl/   
3152   18857  2010       https://are.home.xs4all.nl/   
1527   18836  2002     https://ecpat.home.xs4all.nl/   
1746    4215  2002     https://ecpat.home.xs4all.nl/   
3192   18841  2011    https://graven.home.xs4all.nl/   
16034   6537  2011    https://graven.home.xs4all.nl/   
8653   18849  2008  https://jwwittev.home.xs4all.nl/   
8655    9314  2008  https://jwwittev.home.xs4all.nl/   
10275  18835  2013    https://koenep.home.xs4all.nl/   
10274   9766  2013    https://koenep.home.xs4all.nl/   

                                             description  
15404                                               <NA>  
3152                                      1993, Hongaars  
1527                          defence for children, 2002  
1746   organ

The duplicate URLs sometimes contained variant information in the description. To avoid any loss of metadata, duplicated URL rows are merged and we join the information that's in the description. Empty cells are ignored. 

In [23]:
# Group records by url
df_meta_clean = df_meta.groupby('url').agg({
    # Each row has a unique ID, here we're deciding to keep the first one for the record
    'ID': 'first',
    # Takes any datatype from the description cell, drops empty cells, converts it to strings and joins using a comma as separator
    'description': lambda x: ' , '.join(x.dropna().astype(str)),
    # Don't forget the year
    'year': 'first'

# URL becomes the index
}).reset_index()

# Check
print(f"Original number of records: {len(df_meta)}")
print(f"Unique URLs in the new dataframe: {len(df_meta_clean)}")

Original number of records: 18848
Unique URLs in the new dataframe: 18837


In [24]:
# test if descriptions are merged
# df_master_meta_clean.query("url == 'https://ecpat.home.xs4all.nl/'")
# test if empty cells are ignored
# df_master_meta_clean.query("url == 'https://graven.home.xs4all.nl/'")

Let's simplify the name.

In [25]:
meta = df_meta_clean

#### 1.4 Index csv
We will now load the CSV that contains information about which websites actually were archived and indexed.

In [26]:
# I cheated a little bit and removed the empty row between the headers and data manually in the csv
df_index = pd.read_csv('../metadata/kb-webcollectie_xs4all.csv',
                        encoding='latin-1',
                        sep=';',
                        header=2,
                        skiprows=0,
                        dtype={'Titel': 'string',
                               'Webadres': 'string',
                               'Selectiedatum': 'string',
                               'Status': 'string',
                               'Speciale webcollectie': 'string'}
                       )
df_index.rename(columns={'Titel': 'title',
                          'Webadres': 'url',
                          'Selectiedatum': 'selection-date',
                          'Status': 'status',
                          'Speciale webcollectie': 'subcollection'},
                 inplace=True)

In [28]:
#check imported dataframe
df_index.head()

,title,url,selection-date,status,subcollection
0,Netkwesties,http://www.netkwesties.nl/,15-10-2007,<NA>,XS4ALL
1,Hans Groenewegen,https://kapiteel.home.xs4all.nl/,4-3-2008,<NA>,XS4ALL
2,Dutch Meteor Society (DMS),http://dmsweb.home.xs4all.nl/,14-9-2012,<NA>,XS4ALL ; Internetarcheologie
3,XS4all,https://www.xs4all.nl/,7-8-2013,<NA>,NL-blogosfeer; XS4ALL
4,Het incompleet verzamelde werk van Francisco v...,https://fvjole.home.xs4all.nl/archief/,30-1-2015,Gestopt,"XS4ALL, Internetarcheologie"


#### 1.5 Dropping duplicates
This dataframe only contained a small number of duplicates. Luckily, they had a similar pattern: every first copy had a richer value for 'subcollection.' So it was decided to always keep the first.

In [32]:
print(f"Total number of records in the index_meta dataframe: {len(df_index)}")
print(f"Unique URLs in the index_meta dataframe: {df_index['url'].nunique()}", "\n")

print(f"Number of duplicate records: {len(df_index)-df_index['url'].nunique()}")

# Test view of all duplicate entries 
# index_duplicates = df_index_meta[df_index_meta.duplicated('url', keep=False)]
# print(index_duplicates.sort_values('url')[:21])

# Test view if duplicates were dropped correctly and we have the desired number of 4
# unique_index = index_duplicates.drop_duplicates('url', keep='first')
# print(unique_index)

df_index_clean = df_index.drop_duplicates('url', keep='first')

# Simplification of name
index = df_index_clean

Total number of records in the index_meta dataframe: 3261
Unique URLs in the index_meta dataframe: 3257 

Number of duplicate records: 4


In [63]:
# Test if correct 'subcollection' was taken
# index.query("url == 'https://gniletfe.home.xs4all.nl/'")

## 2. Creating new metadata by intersecting
Now, we see that there had been far more websites considered for the collection, but only a few were selected, allowed **or** possible to harvest.

In [36]:
print(f"Total URLs in the metadata csv: {len(meta)}")
print(f"Total URLs in the archive: {len(index)}")

Total URLs in the metadata csv: 18837
Total URLs in the archive: 3257


### 2.1 Crunching the numbers

In [64]:
print("Legenda:\n Arch(ived) = URL present in index CSV\n Ann(otated) = URL present in meta CSV\n Year value Y/N = missing (N) or present (Y) value in the column 'year' of the meta csv\n")
print("Numbers...")
# 0. Totals
print(f"0. Ann(Y): {len(meta)}")
print(f"0. Arch(Y): {len(index)}")

# 1. Archived and annotated URLs
df_arch_an = meta[meta['url'].isin(index['url'])]
print(f"1. Arch(Y) Ann(Y):", len(df_arch_an))
# If you wish to save this subset, uncomment below
#df_arch_an.to_csv('archived-and-annotated.csv', index=False)

# 2. Archived but not annotated URLS
df_arch_non_an = index[~index['url'].isin(meta['url'])]
print(f"2. Arch(Y) Ann(N):", len(df_arch_non_an))
# If you wish to save this subset, uncomment below
#df_arch_non_an.to_csv('archived-not-annotated.csv', index=False)

# To check for duplicates or missing urls
# print(f"There is a surplus of", (len(df_arch_non_an)+len(df_arch_an))-len(index), "records when we sum the df archived AND annotated URLs and df archived NOT annotated",
#     "\n", "with respect to the total number of archived records in index, which has", len(index), "records.")

# 3. Archived, annotated, but no date
df_no_date = df_arch_an[df_arch_an['year'].isna()]
print(f"3. Arch(Y) Ann(Y) Year value(N):", len(df_no_date))

# 4. Archived, annotated, has year
df_arch_date = df_arch_an.dropna(subset=['year'])
print(f"4. Arch(Y) Ann(Y) Year value (Y):", len(df_arch_date))

Legenda:
 Arch(ived) = URL present in index CSV
 Ann(otated) = URL present in meta CSV
 Year value Y/N = missing (N) or present (Y) value in the column 'year' of the meta csv

Numbers...
0. Ann(Y): 18837
0. Arch(Y): 3257
1. Arch(Y) Ann(Y): 3179
2. Arch(Y) Ann(N): 78
3. Arch(Y) Ann(Y) Year value(N): 22
4. Arch(Y) Ann(Y) Year value (Y): 3157


In [71]:
#quick manual check
#print(df_no_date[:10])

## 3. Preparing the JSON training files
We can do a lot with this information. The _year_ attributed will become important later on in the process, e.g. for some early comparisons after the labeling functions (LF) step with the annotated year and the period bucket a website will fall into. 

In the steps below, we will create some deliverables that serve different purposes:
1. The (new) URL index in JSON format
2. A DF with descriptions for the Knowledge Graph

### 3.1 The URL index in JSON format
Unfortunately, I must be frank about a mistake made early in the process, which, due to time limits, is impossible to fix within the scope of this project. Initially, I excluded the non-annotated websites as at first I was interested in using the year variable as a ground truth. This led to a cut of 78 websites in the eventual training dataset. Requesting WARCs takes time, which there is none left. So below you find the truthful index used for my analysis and the suggestion I would give for reproducing the project.

In [80]:
new_index = df_arch_date[['ID', 'year', 'url']].copy()
print(len(new_index))

3157


In [81]:
# Include also the non-annotated warcs and concatinate dfs for proper column names
df_combined = pd.concat([df_arch_an, df_arch_non_an], ignore_index=True)
better_index = df_combined[['ID', 'year', 'url']].copy()

# Store as a list to request WARCs in the KB
# better_index.to_csv('xs4all_warcs_request.csv', index=False)

Now let's make the dictionary to make the iterative process of checking if a WARC contains the URL of our dataset quicker.

In [82]:
# We don't need the ID information anymore
dic = new_index[['url', 'year']].copy()

# Remove http characters for easier warc processing later on
dic['url'] = dic['url'].str.replace(r'^https://|/$', '', regex=True)
dic['url'] = dic['url'].str.replace(r'^http://|/$', '', regex=True)

index_dic = dict(zip(dic['url'], dic['year'].astype(int)))

In [79]:
import json

# Save it
with open("../warcs/index_warcs.json", "w") as outfile:
    json.dump(index_dic, outfile)

We now created our index! Websites contain many links and thus (external) urls. This can cause the parser to create json lines for urls for subpages or non-archived websites. To give the parser this seed url list to help the parser focus on our target dataset. 

From here on, you can continue to the python scripts in the directory ```/scripts```, but in the blocks below we explore the data a little further.

### 3.2 Dataframe for Knowledge Graph

For now, we strand this step for a little bit until I have more knowledge on how to process the information from the WCT.

## Inference set

These webpages don't have a date annotation. We will use them later as an inference set when the model is trained.

In [ ]:
# In the web archive but not mentioned in the masterbestand, hence no annotation nor date available
non_an = df_arch_non_an
# In the archive, mentioned in masterbestand, but missing value of date
non_date = df_no_date

inference = [non_an, non_date]
inference = pd.concat(inference)
inference.info()

In [ ]:
inf_url_year = inference[['url', 'year']]
inf_url_year.to_csv('inference_warc_set.csv', index=False)

inference_dic = dict(zip(inf_url_year['url'], inf_url_year['year'].fillna(0).astype(int)))